# 🍽️ Hangout Place Recommendation System — Bangalore
**Dataset:** Zomato Bangalore Restaurants (~51,000 records)  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Scikit-learn · SQL  
**Author:** Somesh | MSc Data Analytics, Christ University

---
### Project Phases
| Phase | Description |
|-------|-------------|
| 1 | Data Loading & Cleaning |
| 2 | Feature Engineering |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Content-Based Recommendation Engine |
| 5 | SQL Filtering Layer |


## Phase 1 — Data Loading & Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['axes.edgecolor'] = '#e94560'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
PALETTE = ['#e94560','#f5a623','#50fa7b','#8be9fd','#bd93f9','#ff79c6','#ffb86c']

print("Libraries imported successfully!")

In [ ]:
# Load dataset
df = pd.read_csv('../data/zomato.csv')
print(f"Raw shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

In [ ]:
# Check nulls and data types
print("Null counts:")
print(df.isnull().sum())
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Drop irrelevant columns
df.drop(columns=['url', 'phone', 'address', 'reviews_list', 'menu_item'], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)
print(f"After cleaning shape: {df.shape}")

# Clean 'rate': '4.1/5' -> 4.1
df['rate'] = df['rate'].astype(str).str.replace('/5', '', regex=False).str.strip()
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')

# Clean cost: remove commas
df['approx_cost(for two people)'] = (
    df['approx_cost(for two people)']
    .astype(str).str.replace(',', '', regex=False).str.strip()
)
df['approx_cost(for two people)'] = pd.to_numeric(
    df['approx_cost(for two people)'], errors='coerce'
)

# Rename columns for convenience
df.rename(columns={
    'approx_cost(for two people)': 'cost_for_two',
    'listed_in(type)': 'listing_type',
    'listed_in(city)': 'city_area'
}, inplace=True)

# Fill nulls
df['rate'].fillna(df['rate'].median(), inplace=True)
df['cost_for_two'].fillna(df['cost_for_two'].median(), inplace=True)
df['rest_type'].fillna('Unknown', inplace=True)
df['cuisines'].fillna('Unknown', inplace=True)
df['dish_liked'].fillna('', inplace=True)
df['location'].fillna('Unknown', inplace=True)

print(f"\nFinal clean shape: {df.shape}")
print(f"\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head(3)

## Phase 2 — Feature Engineering

In [ ]:
# Budget buckets
def budget_bucket(cost):
    if cost <= 300:   return 'Budget (<=300)'
    elif cost <= 600: return 'Mid-Range (301-600)'
    elif cost <= 1000: return 'Premium (601-1000)'
    else:             return 'Luxury (1000+)'

df['budget_category'] = df['cost_for_two'].apply(budget_bucket)
print("Budget categories:")
print(df['budget_category'].value_counts())

In [ ]:
# Occasion tags
def occasion_tag(row):
    lt = str(row['listing_type']).lower()
    rt = str(row['rest_type']).lower()
    tags = []
    if any(x in lt for x in ['dine-out', 'buffet']) or 'fine dining' in rt:
        tags.append('Date Night')
    if any(x in lt for x in ['cafe', 'desserts', 'drinks']):
        tags.append('Friends Hangout')
    if any(x in lt for x in ['buffet', 'dine-out']) or 'casual dining' in rt:
        tags.append('Family Outing')
    if any(x in lt for x in ['pubs', 'nightlife', 'drinks']):
        tags.append('Night Out')
    if any(x in rt for x in ['quick bites', 'delivery', 'mess']):
        tags.append('Solo / Quick Meal')
    if not tags:
        tags.append('General')
    return ', '.join(tags)

df['occasion_tags'] = df.apply(occasion_tag, axis=1)
print("Sample occasion tags:")
print(df['occasion_tags'].value_counts().head(8))

In [ ]:
# Ambience mapping
def ambience_map(rest_type):
    rt = str(rest_type).lower()
    if 'fine dining' in rt:          return 'Upscale / Romantic'
    elif 'casual dining' in rt:      return 'Casual / Social'
    elif 'cafe' in rt:               return 'Cozy / Chill'
    elif 'pub' in rt or 'bar' in rt: return 'Lively / Party'
    elif 'quick bites' in rt or 'delivery' in rt: return 'Fast / Functional'
    elif 'dessert' in rt or 'bakery' in rt: return 'Sweet / Relaxed'
    else:                            return 'General'

df['ambience'] = df['rest_type'].apply(ambience_map)
print("Ambience distribution:")
print(df['ambience'].value_counts())

In [ ]:
# Preview engineered features
df[['name','location','cost_for_two','budget_category','occasion_tags','ambience']].head(10)

## Phase 3 — Exploratory Data Analysis (EDA)

In [ ]:
# EDA — 9 Panel Dashboard
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.patch.set_facecolor('#0f0f0f')
fig.suptitle('Zomato Bangalore — Hangout Recommendation EDA',
             fontsize=20, color='white', fontweight='bold', y=1.01)

ACCENT = '#e94560'

# 1. Top 10 Locations
ax = axes[0, 0]
top_loc = df['location'].value_counts().head(10)
bars = ax.barh(top_loc.index[::-1], top_loc.values[::-1], color=PALETTE[0])
ax.set_title('Top 10 Locations', color='white', fontweight='bold')
ax.set_xlabel('Count', color='white')
for bar, val in zip(bars, top_loc.values[::-1]):
    ax.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2,
            str(val), va='center', color='white', fontsize=8)

# 2. Rating Distribution
ax = axes[0, 1]
ax.hist(df['rate'], bins=30, color=PALETTE[1], edgecolor='#0f0f0f', alpha=0.9)
ax.axvline(df['rate'].mean(), color=ACCENT, linestyle='--', linewidth=2,
           label=f"Mean: {df['rate'].mean():.2f}")
ax.set_title('Rating Distribution', color='white', fontweight='bold')
ax.set_xlabel('Rating', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

# 3. Cost Distribution
ax = axes[0, 2]
cost_f = df[df['cost_for_two'] <= 2000]['cost_for_two']
ax.hist(cost_f, bins=40, color=PALETTE[2], edgecolor='#0f0f0f', alpha=0.9)
ax.axvline(cost_f.median(), color=ACCENT, linestyle='--', linewidth=2,
           label=f"Median: Rs.{cost_f.median():.0f}")
ax.set_title('Cost for Two Distribution', color='white', fontweight='bold')
ax.set_xlabel('Cost (Rs.)', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

# 4. Budget Category Pie
ax = axes[1, 0]
bc = df['budget_category'].value_counts()
wedges, texts, autotexts = ax.pie(bc.values, labels=bc.index, autopct='%1.1f%%',
    colors=PALETTE[:4], textprops={'color':'white'}, startangle=140)
for a in autotexts: a.set_color('white')
ax.set_title('Budget Categories', color='white', fontweight='bold')

# 5. Listing Types
ax = axes[1, 1]
lt = df['listing_type'].value_counts()
bars = ax.bar(lt.index, lt.values, color=PALETTE[:len(lt)])
ax.set_title('Listing Types (Occasion)', color='white', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, lt.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            str(val), ha='center', color='white', fontsize=8)

# 6. Ambience Distribution
ax = axes[1, 2]
amb = df['ambience'].value_counts()
ax.barh(amb.index[::-1], amb.values[::-1], color=PALETTE[4])
ax.set_title('Ambience Types', color='white', fontweight='bold')
ax.set_xlabel('Count', color='white')

# 7. Online Order vs Rating
ax = axes[2, 0]
df.boxplot(column='rate', by='online_order', ax=ax,
    boxprops=dict(color=ACCENT), whiskerprops=dict(color='white'),
    medianprops=dict(color=PALETTE[2], linewidth=2),
    capprops=dict(color='white'),
    flierprops=dict(color=ACCENT, marker='o', markersize=2))
ax.set_title('Rating by Online Order', color='white', fontweight='bold')
ax.set_xlabel('Online Order', color='white')
plt.sca(ax); plt.title('Rating by Online Order', color='white')

# 8. Top 10 Cuisines
ax = axes[2, 1]
cuisine_series = df['cuisines'].str.split(', ').explode()
top_c = cuisine_series.value_counts().head(10)
ax.barh(top_c.index[::-1], top_c.values[::-1], color=PALETTE[5])
ax.set_title('Top 10 Cuisines', color='white', fontweight='bold')
ax.set_xlabel('Count', color='white')

# 9. Avg Rating by Budget
ax = axes[2, 2]
budget_order = ['Budget (<=300)','Mid-Range (301-600)','Premium (601-1000)','Luxury (1000+)']
avg_r = df.groupby('budget_category')['rate'].mean().reindex(budget_order)
bars = ax.bar(range(4), avg_r.values, color=PALETTE[:4])
ax.set_xticks(range(4))
ax.set_xticklabels(['Budget','Mid','Premium','Luxury'], rotation=15, color='white')
ax.set_title('Avg Rating by Budget', color='white', fontweight='bold')
ax.set_ylabel('Avg Rating', color='white')
for bar, val in zip(bars, avg_r.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.2f}', ha='center', color='white', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/zomato_eda.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print("EDA chart saved to data/zomato_eda.png")

### Key EDA Insights

In [ ]:
# Quick stats summary
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total restaurants       : {len(df):,}")
print(f"Unique locations        : {df['location'].nunique()}")
print(f"Unique cuisines         : {df['cuisines'].str.split(', ').explode().nunique()}")
print(f"Average rating          : {df['rate'].mean():.2f} / 5")
print(f"Median cost for two     : Rs. {df['cost_for_two'].median():.0f}")
print(f"Online order available  : {(df['online_order']=='Yes').sum():,} restaurants")
print(f"Table booking available : {(df['book_table']=='Yes').sum():,} restaurants")
print()
print("Top 5 locations:")
print(df['location'].value_counts().head(5).to_string())
print()
print("Top 5 cuisines:")
print(df['cuisines'].str.split(', ').explode().value_counts().head(5).to_string())

## Phase 4 — Content-Based Recommendation Engine

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build combined feature string per restaurant
def build_features(row):
    parts = [
        str(row['cuisines']),
        str(row['rest_type']),
        str(row['listing_type']),
        str(row['ambience']),
        str(row['budget_category']),
        str(row['dish_liked'])
    ]
    return ' '.join(parts).lower()

df['features'] = df.apply(build_features, axis=1)

# Fit TF-IDF
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['features'])

df.reset_index(drop=True, inplace=True)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print("Recommendation engine ready!")

In [ ]:
def recommend(
    occasion='Date Night',
    budget='Mid-Range (301-600)',
    cuisine=None,
    location=None,
    top_n=10
):
    """
    Recommend hangout places based on preferences.

    Parameters
    ----------
    occasion : str
        'Date Night' | 'Friends Hangout' | 'Family Outing' | 'Night Out' | 'Solo / Quick Meal'
    budget : str
        'Budget (<=300)' | 'Mid-Range (301-600)' | 'Premium (601-1000)' | 'Luxury (1000+)'
    cuisine : str, optional
        e.g. 'North Indian', 'Chinese', 'Italian', 'Continental'
    location : str, optional
        Bangalore area e.g. 'Indiranagar', 'Koramangala 5th Block', 'HSR'
    top_n : int
        Number of recommendations to return
    """
    query_parts = [occasion, budget]
    if cuisine:  query_parts.append(cuisine)
    query = ' '.join(query_parts).lower()

    query_vec  = tfidf.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    filtered = df.copy()
    filtered['sim_score'] = sim_scores

    if location:
        filtered = filtered[
            filtered['location'].str.lower().str.contains(location.lower(), na=False)
        ]
    if budget:
        filtered = filtered[filtered['budget_category'] == budget]

    filtered['final_score'] = (
        filtered['sim_score'] * 0.6 +
        (filtered['rate'] / 5) * 0.4
    )

    result = (filtered
              .sort_values('final_score', ascending=False)
              .drop_duplicates(subset='name')
              .head(top_n)
              [['name','location','cuisines','rate','votes',
                'cost_for_two','ambience','budget_category',
                'online_order','book_table','occasion_tags']])
    return result

print("recommend() function defined!")

### Test the Recommender

In [ ]:
# Test 1: Date Night — Premium
print("=" * 60)
print("Date Night | Premium | Italian")
print("=" * 60)
result = recommend(
    occasion='Date Night',
    budget='Premium (601-1000)',
    cuisine='Italian',
    top_n=5
)
result[['name','location','cuisines','rate','cost_for_two','ambience']]

In [ ]:
# Test 2: Friends Hangout in Indiranagar
print("=" * 60)
print("Friends Hangout | Mid-Range | Indiranagar")
print("=" * 60)
result = recommend(
    occasion='Friends Hangout',
    budget='Mid-Range (301-600)',
    location='Indiranagar',
    top_n=5
)
result[['name','location','cuisines','rate','cost_for_two','ambience']]

In [ ]:
# Test 3: Family outing, budget North Indian
print("=" * 60)
print("Family Outing | Budget | North Indian")
print("=" * 60)
result = recommend(
    occasion='Family Outing',
    budget='Budget (<=300)',
    cuisine='North Indian',
    top_n=5
)
result[['name','location','cuisines','rate','cost_for_two','ambience']]

In [ ]:
# Test 4: Night Out — Koramangala
print("=" * 60)
print("Night Out | Mid-Range | Koramangala")
print("=" * 60)
result = recommend(
    occasion='Night Out',
    budget='Mid-Range (301-600)',
    location='Koramangala',
    top_n=5
)
result[['name','location','cuisines','rate','cost_for_two','ambience']]

## Phase 5 — SQL Filtering Layer
Using Python's `sqlite3` to create an in-memory SQL database for structured queries on top of the recommendation engine.


In [ ]:
import sqlite3

# Load cleaned data into SQLite
conn = sqlite3.connect(':memory:')
df.to_sql('restaurants', conn, index=False, if_exists='replace')
print("SQLite database created with 'restaurants' table.")
print(f"Rows loaded: {pd.read_sql('SELECT COUNT(*) as total FROM restaurants', conn).iloc[0,0]:,}")

In [ ]:
# SQL Query 1: Best rated budget restaurants
query1 = '''
SELECT name, location, cuisines, rate, cost_for_two, ambience
FROM restaurants
WHERE budget_category = 'Budget (<=300)'
  AND rate >= 4.0
  AND online_order = 'Yes'
ORDER BY rate DESC, votes DESC
LIMIT 10
'''
pd.read_sql(query1, conn)

In [ ]:
# SQL Query 2: Date Night spots in Koramangala with table booking
query2 = '''
SELECT name, location, cuisines, rate, cost_for_two, ambience
FROM restaurants
WHERE occasion_tags LIKE '%Date Night%'
  AND location LIKE '%Koramangala%'
  AND book_table = 'Yes'
ORDER BY rate DESC
LIMIT 10
'''
pd.read_sql(query2, conn)

In [ ]:
# SQL Query 3: Top cuisines by average rating
query3 = '''
SELECT cuisines, 
       ROUND(AVG(rate), 2) AS avg_rating,
       COUNT(*) AS total_restaurants
FROM restaurants
WHERE cuisines != 'Unknown'
GROUP BY cuisines
HAVING total_restaurants > 20
ORDER BY avg_rating DESC
LIMIT 15
'''
pd.read_sql(query3, conn)

In [ ]:
# SQL Query 4: Location summary - avg cost and rating
query4 = '''
SELECT location,
       COUNT(*) AS total,
       ROUND(AVG(rate), 2) AS avg_rating,
       ROUND(AVG(cost_for_two), 0) AS avg_cost
FROM restaurants
GROUP BY location
HAVING total > 100
ORDER BY avg_rating DESC
LIMIT 15
'''
pd.read_sql(query4, conn)

## Save Clean Dataset

In [ ]:
# Save the cleaned + engineered dataset
df.to_csv('../data/zomato_clean.csv', index=False)
print(f"Saved zomato_clean.csv — shape: {df.shape}")
print("\nFinal columns:")
for col in df.columns:
    print(f"  {col}")

---
## Project Summary

| Phase | Status | Output |
|-------|--------|--------|
| 1. Data Cleaning | ✅ Done | 51,609 clean records |
| 2. Feature Engineering | ✅ Done | budget_category, occasion_tags, ambience |
| 3. EDA | ✅ Done | 9-panel dashboard |
| 4. ML Recommender | ✅ Done | TF-IDF content-based engine |
| 5. SQL Layer | ✅ Done | SQLite in-memory queries |

**Next Steps:**
- Build a Streamlit web app for interactive recommendations
- Add collaborative filtering using votes/ratings
- Deploy to Streamlit Cloud or Hugging Face Spaces
